# VRP Cross-Sectional Factor Model
**Replication & Extension of Carr-Wu (2009)**

This notebook is fully self-contained. Just run every cell top-to-bottom (`Shift+Enter`).

- **Cell 1** installs all packages automatically
- **Cell 2** pulls real-time S&P 500 data from Yahoo Finance
- **Cells 3–8** build the VRP factor model, run Fama-MacBeth regressions, and plot results

> `VRP = IV² − RV²`  where IV² = implied variance (from VIX proxy), RV² = realised variance from daily returns

In [ ]:
# ── CELL 1 · Auto-install all dependencies ─────────────────────────────────
import subprocess, sys

pkgs = ['yfinance', 'pandas', 'numpy', 'scipy', 'statsmodels',
        'matplotlib', 'seaborn', 'pandas_datareader']

for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✓ All packages installed')

In [ ]:
# ── CELL 2 · Pull real-time data from Yahoo Finance ────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── S&P 500 large-cap universe (50 liquid names across sectors) ──
TICKERS = [
    # Tech
    'AAPL','MSFT','NVDA','GOOGL','META','AMZN','TSLA','AVGO','ORCL','CRM',
    # Financials
    'JPM','BAC','WFC','GS','MS','BLK','AXP','SPGI','CB','PGR',
    # Healthcare
    'LLY','UNH','JNJ','ABBV','MRK','TMO','ABT','DHR','SYK','ISRG',
    # Consumer
    'WMT','COST','PG','KO','PEP','MCD','NKE','TGT','HD','LOW',
    # Energy / Industrials
    'XOM','CVX','CAT','HON','UNP','RTX','LMT','GE','MMM','BA'
]

print(f'Downloading {len(TICKERS)} tickers — this takes ~30 seconds...')

# Daily prices: 10 years of history
raw = yf.download(TICKERS, period='10y', interval='1d',
                  auto_adjust=True, progress=False)

prices = raw['Close'].dropna(how='all', axis=1)
TICKERS = prices.columns.tolist()
print(f'✓ Got {len(TICKERS)} tickers × {len(prices)} trading days')
print(f'  Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

# VIX as implied volatility proxy
print('Downloading VIX...')
vix_raw = yf.download('^VIX', period='10y', interval='1d',
                      auto_adjust=True, progress=False)
vix = vix_raw['Close'].squeeze()
print(f'✓ VIX loaded: {len(vix)} days')

In [ ]:
# ── CELL 3 · Compute IV², RV², and VRP ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 12,
    'font.family': 'monospace'
})

LOOKBACK = 21   # trading days for RV window
FORWARD  = 21   # trading days for forward return

# ── Daily log returns ──
log_ret = np.log(prices / prices.shift(1))

# ── RV²: rolling 21-day realised variance (annualised) ──
rv2_daily = (log_ret ** 2).rolling(LOOKBACK).sum() * (252 / LOOKBACK)

# ── IV²: use VIX²/10000 as market IV proxy, spread to stocks via beta ──
# VIX is quoted in %, so VIX=20 → σ=0.20 → variance=0.04
vix_var = (vix / 100) ** 2   # annualised implied variance

# Rolling 60-day beta of each stock vs SPY
spy_raw = yf.download('SPY', period='10y', interval='1d',
                       auto_adjust=True, progress=False)
spy_ret = np.log(spy_raw['Close'].squeeze() / spy_raw['Close'].squeeze().shift(1))

betas = pd.DataFrame(index=log_ret.index, columns=TICKERS, dtype=float)
for ticker in TICKERS:
    cov = log_ret[ticker].rolling(60).cov(spy_ret)
    var = spy_ret.rolling(60).var()
    betas[ticker] = (cov / var).clip(0.2, 3.0)

# IV² per stock: β² × market IV² (simplified single-factor)
iv2_daily = betas ** 2 * vix_var.reindex(log_ret.index).fillna(method='ffill')

# ── VRP per stock per day ──
vrp_daily = iv2_daily - rv2_daily

# ── Resample to monthly (month-end) ──
def to_monthly(df):
    return df.resample('ME').last()

vrp_m   = to_monthly(vrp_daily)
rv2_m   = to_monthly(rv2_daily)
iv2_m   = to_monthly(iv2_daily)
beta_m  = to_monthly(betas)

# Monthly log returns (forward: next month's return)
prices_m   = to_monthly(prices)
ret_m      = np.log(prices_m / prices_m.shift(1))
ret_fwd_m  = ret_m.shift(-1)   # next-month return

# Market cap proxy: price × shares (use price as rank proxy, fine cross-sectionally)
log_me_m = np.log(prices_m)

print(f'✓ Monthly panels built: {vrp_m.shape[0]} months × {vrp_m.shape[1]} stocks')
print(f'  VRP mean: {vrp_m.mean().mean():.4f}  (positive = market pays variance insurance premium)')

# ── Plot 1: Average VRP over time ──
fig, ax = plt.subplots(figsize=(12, 3))
vrp_avg = vrp_m.mean(axis=1)
ax.fill_between(vrp_avg.index, 0, vrp_avg, where=vrp_avg>=0,
                color='#3B6D11', alpha=0.6, label='Positive VRP')
ax.fill_between(vrp_avg.index, 0, vrp_avg, where=vrp_avg<0,
                color='#A32D2D', alpha=0.6, label='Negative VRP')
ax.axhline(0, color='#444441', linewidth=0.8)
ax.set_title('Average cross-sectional VRP over time  (IV² − RV²)')
ax.set_ylabel('Annualised variance')
ax.legend()
plt.tight_layout()
plt.show()
print('↑ Higher VRP months = market pricing in more variance risk')

In [ ]:
# ── CELL 4 · Fama-MacBeth cross-sectional regressions ─────────────────────
from scipy import stats

def newey_west_tstat(series, lags=4):
    """Mean / Newey-West SE for a time series of coefficient estimates."""
    s = series.dropna()
    n = len(s)
    if n < 10:
        return np.nan, np.nan, n
    mean = s.mean()
    d = s - mean
    v = (d**2).mean()
    for lag in range(1, lags+1):
        w = 1 - lag / (lags + 1)
        v += 2 * w * (d[lag:].values * d[:-lag].values).mean()
    se = np.sqrt(max(v, 1e-12) / n)
    return float(mean), float(mean / se), n

def run_fm(dates_idx):
    """Run FM regression over a subset of dates, return slope on VRP."""
    lambdas = []
    for date in dates_idx:
        y = ret_fwd_m.loc[date].dropna()
        x_vrp  = vrp_m.loc[date].reindex(y.index)
        x_beta = beta_m.loc[date].reindex(y.index)
        x_me   = log_me_m.loc[date].reindex(y.index)

        df = pd.DataFrame({'ret': y, 'vrp': x_vrp,
                           'beta': x_beta, 'logme': x_me}).dropna()
        if len(df) < 15:
            continue

        # Winsorise at 1%/99%
        for col in ['vrp','beta','logme']:
            lo, hi = df[col].quantile([0.01, 0.99])
            df[col] = df[col].clip(lo, hi)

        X = np.column_stack([np.ones(len(df)), df['vrp'], df['beta'], df['logme']])
        try:
            coefs, *_ = np.linalg.lstsq(X, df['ret'].values, rcond=None)
            lambdas.append(coefs[1])   # slope on VRP
        except np.linalg.LinAlgError:
            continue

    return pd.Series(lambdas)

# Full date index
all_dates = vrp_m.index.intersection(ret_fwd_m.index).intersection(beta_m.index)

# Split into sub-periods
sub_periods = {
    'Pre-2018 (early)':   all_dates[all_dates < '2018-01-01'],
    '2018-2021':          all_dates[(all_dates>='2018-01-01')&(all_dates<'2022-01-01')],
    '2022-present':       all_dates[all_dates >= '2022-01-01'],
    'Full sample':        all_dates,
}

print('Running Fama-MacBeth regressions...\n')
print(f'{"Period":<22} {"λ₁(VRP)":>10} {"t-stat":>8} {"N months":>9} {"Signal"}')
print('─' * 65)

fm_results = {}
for label, idx in sub_periods.items():
    lams = run_fm(idx)
    mean, tstat, n = newey_west_tstat(lams)
    fm_results[label] = {'lambda': mean, 't_stat': tstat, 'n': n}
    sig = ('✅ strong'  if abs(tstat)>2.5 else
           '⚠️  moderate' if abs(tstat)>1.65 else
           '❌ weak')
    sign = '+' if mean >= 0 else ''
    print(f'{label:<22} {sign}{mean:>9.5f} {tstat:>8.2f} {n:>9}  {sig}')

print('\nNewey-West HAC standard errors (4 lags). Controls: beta, log(ME).')

In [ ]:
# ── CELL 5 · Rolling λ₁ — visualise decay ─────────────────────────────────
ROLL_WIN = 24   # months

rolling_means, rolling_tstats, roll_dates = [], [], []
for i in range(ROLL_WIN, len(all_dates)):
    window_dates = all_dates[i-ROLL_WIN:i]
    lams = run_fm(window_dates)
    mean, tstat, n = newey_west_tstat(lams)
    rolling_means.append(mean)
    rolling_tstats.append(tstat)
    roll_dates.append(all_dates[i])

roll_df = pd.DataFrame({'lambda': rolling_means, 'tstat': rolling_tstats},
                        index=pd.DatetimeIndex(roll_dates))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Top: rolling lambda
axes[0].plot(roll_df.index, roll_df['lambda'], color='#185FA5', linewidth=1.5)
axes[0].fill_between(roll_df.index, 0, roll_df['lambda'],
                     where=roll_df['lambda']>=0, alpha=0.15, color='#185FA5')
axes[0].axhline(0, color='#444441', linewidth=0.7, linestyle='--')
axes[0].set_ylabel('λ₁ (VRP slope)')
axes[0].set_title(f'Rolling {ROLL_WIN}-month Fama-MacBeth VRP coefficient')

# Bottom: t-stat with significance band
axes[1].plot(roll_df.index, roll_df['tstat'], color='#BA7517', linewidth=1.5)
axes[1].axhline(1.96,  color='#3B6D11', linewidth=0.8, linestyle='--', label='5% (±1.96)')
axes[1].axhline(-1.96, color='#3B6D11', linewidth=0.8, linestyle='--')
axes[1].axhline(0,     color='#444441', linewidth=0.5, linestyle=':')
axes[1].fill_between(roll_df.index, -1.96, 1.96, alpha=0.06, color='#888780')
axes[1].set_ylabel('t-statistic')
axes[1].legend()

plt.tight_layout()
plt.savefig('rolling_lambda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → rolling_lambda.png')

In [ ]:
# ── CELL 6 · Quintile portfolio sorts ─────────────────────────────────────
Q = 5
port_rets = {f'Q{i+1}': [] for i in range(Q)}
port_rets['LS'] = []
port_dates = []

for date in all_dates[:-1]:
    sig  = vrp_m.loc[date].dropna()
    fwd  = ret_fwd_m.loc[date].reindex(sig.index).dropna()
    common = sig.index.intersection(fwd.index)
    if len(common) < Q * 5:
        continue

    sig, fwd = sig.loc[common], fwd.loc[common]
    # Price as market-cap proxy (value-weighting)
    mc = prices_m.loc[date].reindex(common).fillna(1)

    ranks = pd.qcut(sig, Q, labels=[f'Q{i+1}' for i in range(Q)])
    port_dates.append(date)

    q_rets = {}
    for q in [f'Q{i+1}' for i in range(Q)]:
        members = ranks[ranks == q].index
        w = mc.loc[members] / mc.loc[members].sum()
        q_rets[q] = float((fwd.loc[members] * w).sum())
        port_rets[q].append(q_rets[q])

    port_rets['LS'].append(q_rets['Q5'] - q_rets['Q1'])

port_df = pd.DataFrame(port_rets, index=pd.DatetimeIndex(port_dates))

# ── Summary stats ──
print(f'\n{"Portfolio":<10} {"Ann. Ret":>10} {"Ann. Vol":>10} {"Sharpe":>8} {"Max DD":>9}')
print('─' * 55)

for col in [f'Q{i+1}' for i in range(Q)] + ['LS']:
    s = port_df[col].dropna()
    ann_ret = s.mean() * 12
    ann_vol = s.std() * np.sqrt(12)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = (1 + s).cumprod()
    max_dd = ((cum - cum.cummax()) / cum.cummax()).min()
    label = f'{col} (high VRP)' if col=='Q5' else (f'{col} (low VRP)' if col=='Q1' else col)
    print(f'{label:<14} {ann_ret*100:>8.2f}%  {ann_vol*100:>8.2f}%  {sharpe:>7.2f}  {max_dd*100:>8.2f}%')

# ── Plot: cumulative returns ──
cum_all = (1 + port_df[[f'Q{i+1}' for i in range(Q)] + ['LS']]).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#A32D2D','#E24B4A','#888780','#639922','#185FA5']
for i, q in enumerate([f'Q{i+1}' for i in range(Q)]):
    ax.plot(cum_all.index, cum_all[q], color=colors[i],
            alpha=0.55, linewidth=1.2, label=q)
ax.plot(cum_all.index, cum_all['LS'], color='#000000',
        linewidth=2.2, linestyle='--', label='L/S (Q5−Q1)')
ax.axhline(1, color='#888780', linewidth=0.5, linestyle=':')
ax.set_title('VRP quintile portfolios — cumulative value of $1')
ax.set_ylabel('Growth of $1')
ax.legend(ncol=3)
plt.tight_layout()
plt.savefig('cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → cumulative_returns.png')

In [ ]:
# ── CELL 7 · Microstructure & decay analysis ───────────────────────────────

# ── RV sampling frequency sensitivity ──
print('=== RV sampling frequency sensitivity ===\n')
for window, label in [(10,'10-day (2-week)'), (21,'21-day (1-month, baseline)'),
                       (63,'63-day (1-quarter)')]:
    rv2_alt = (log_ret**2).rolling(window).sum() * (252/window)
    vrp_alt = iv2_daily - rv2_alt
    vrp_alt_m = to_monthly(vrp_alt)

    # Quick FM using full sample
    lams = []
    for date in all_dates[:-1]:
        y = ret_fwd_m.loc[date].dropna()
        x = vrp_alt_m.loc[date].reindex(y.index)
        df = pd.DataFrame({'ret': y, 'vrp': x}).dropna()
        if len(df) < 10: continue
        lo, hi = df['vrp'].quantile([0.01,0.99])
        df['vrp'] = df['vrp'].clip(lo, hi)
        X = np.column_stack([np.ones(len(df)), df['vrp']])
        try:
            c, *_ = np.linalg.lstsq(X, df['ret'].values, rcond=None)
            lams.append(c[1])
        except: pass
    mean, tstat, n = newey_west_tstat(pd.Series(lams))
    print(f'  {label:<26}  λ₁={mean:+.5f}  t={tstat:.2f}')

# ── Crisis vs non-crisis ──
print('\n=== Crisis-period dependency ===\n')
crisis = (
    ((all_dates >= '2020-02-01') & (all_dates <= '2020-06-30')) |
    ((all_dates >= '2022-01-01') & (all_dates <= '2022-12-31'))
)
for label, mask in [('Full sample', pd.Series([True]*len(all_dates), index=all_dates)),
                     ('Excl. crisis', ~pd.Series(crisis, index=all_dates))]:
    idx = all_dates[mask.values]
    lams = run_fm(idx)
    mean, tstat, n = newey_west_tstat(lams)
    print(f'  {label:<20}  λ₁={mean:+.5f}  t={tstat:.2f}  N={n}')

# ── VRP distribution today ──
latest_date = vrp_m.index[-1]
latest_vrp  = vrp_m.loc[latest_date].dropna().sort_values()

print(f'\n=== Current VRP snapshot ({latest_date.date()}) ===')
print(f'  Mean VRP:    {latest_vrp.mean():.4f}')
print(f'  Median VRP:  {latest_vrp.median():.4f}')
print(f'  % positive:  {(latest_vrp>0).mean()*100:.1f}%')
print(f'\n  Top 5 high-VRP stocks (long candidates):')
print(latest_vrp.tail(5).to_frame('VRP').to_string())
print(f'\n  Bottom 5 low-VRP stocks (short candidates):')
print(latest_vrp.head(5).to_frame('VRP').to_string())

In [ ]:
# ── CELL 8 · Summary dashboard plot ───────────────────────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.35)

# 1. Average VRP over time
ax1 = fig.add_subplot(gs[0, :])
vrp_ts = vrp_m.mean(axis=1).dropna()
ax1.fill_between(vrp_ts.index, 0, vrp_ts,
                 where=vrp_ts>=0, color='#3B6D11', alpha=0.55)
ax1.fill_between(vrp_ts.index, 0, vrp_ts,
                 where=vrp_ts<0,  color='#A32D2D', alpha=0.55)
ax1.axhline(0, color='#444441', linewidth=0.7)
ax1.set_title('Cross-sectional mean VRP (IV² − RV²)  — live data')
ax1.set_ylabel('Annualised variance premium')

# 2. Quintile spread — bar
ax2 = fig.add_subplot(gs[1, 0])
q_means = [port_df[f'Q{i+1}'].mean()*12*100 for i in range(Q)]
bars = ax2.bar([f'Q{i+1}' for i in range(Q)], q_means,
               color=['#A32D2D','#E24B4A','#888780','#639922','#185FA5'])
ax2.axhline(0, color='#444441', linewidth=0.6)
ax2.set_title('Ann. return by VRP quintile')
ax2.set_ylabel('Return (%)')

# 3. Rolling t-stat
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(roll_df.index, roll_df['tstat'], color='#BA7517', linewidth=1.4)
ax3.axhline(1.96,  color='#3B6D11', linewidth=0.8, linestyle='--')
ax3.axhline(-1.96, color='#3B6D11', linewidth=0.8, linestyle='--')
ax3.axhline(0,     color='#444441', linewidth=0.5)
ax3.fill_between(roll_df.index, -1.96, 1.96, alpha=0.07, color='#888780')
ax3.set_title(f'Rolling {ROLL_WIN}-mo t-stat on VRP')
ax3.set_ylabel('t-statistic')

# 4. Current VRP distribution
ax4 = fig.add_subplot(gs[1, 2])
ax4.hist(latest_vrp, bins=15, color='#185FA5', alpha=0.75, edgecolor='white')
ax4.axvline(latest_vrp.median(), color='#A32D2D', linewidth=1.5,
            linestyle='--', label=f'Median {latest_vrp.median():.3f}')
ax4.set_title(f'VRP distribution ({latest_date.date()})')
ax4.set_xlabel('VRP')
ax4.legend(fontsize=9)

plt.suptitle('VRP Factor Model — Carr-Wu (2009) replication with live data',
             fontsize=13, fontweight='normal', y=1.01)
plt.savefig('vrp_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ All done! Saved → vrp_dashboard.png')
print(f'  Data is live through {prices.index[-1].date()}')